In [ ]:
!pip install -q "transformers>=4.45.0" "peft>=0.13.0" datasets accelerate evaluate scikit-learn safetensors torch

In [8]:
from datasets import load_dataset
from collections import Counter
import json

ds = load_dataset("TIGER-Lab/MMLU-Pro")

# 与现有 category_mapping.json 一致
CATEGORY_TO_IDX = {
    "biology": 0, "business": 1, "chemistry": 2, "computer science": 3,
    "economics": 4, "engineering": 5, "health": 6, "history": 7,
    "law": 8, "math": 9, "other": 10, "philosophy": 11,
    "physics": 12, "psychology": 13
}
IDX_TO_CATEGORY = {v: k for k, v in CATEGORY_TO_IDX.items()}

def normalize_category(cat):
    """MMLU-Pro 用 'computer_science'，映射表用 'computer science'"""
    return cat.replace("_", " ")

def preprocess(example):
    example["label"] = CATEGORY_TO_IDX[normalize_category(example["category"])]
    options_text = " ".join(
        [f"({chr(65+i)}) {opt}" for i, opt in enumerate(example["options"]) if opt]
    )
    example["text"] = f"{example['question']} {options_text}"
    return example

# test 集 12032 条用于训练，validation 集 70 条用于验证
train_ds = ds["test"].map(preprocess)
eval_ds = ds["validation"].map(preprocess)

label_counts = Counter(train_ds["label"])
for idx in sorted(label_counts.keys()):
    print(f"  {IDX_TO_CATEGORY[idx]}: {label_counts[idx]}")

  biology: 717
  business: 789
  chemistry: 1132
  computer science: 410
  economics: 844
  engineering: 969
  health: 818
  history: 381
  law: 1101
  math: 1351
  other: 924
  philosophy: 499
  physics: 1299
  psychology: 798


In [9]:
from transformers import AutoTokenizer, ModernBertForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
import torch

BASE_MODEL_ID = "llm-semantic-router/mmbert-32k-yarn"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

model = ModernBertForSequenceClassification.from_pretrained(
    BASE_MODEL_ID,
    num_labels=14,
    id2label=IDX_TO_CATEGORY,
    label2id=CATEGORY_TO_IDX,
    torch_dtype=torch.float32,
)

# 与项目中现有分类器的 LoRA 配置一致
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=["attn.Wqkv", "attn.Wo", "mlp.Wi", "mlp.Wo"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: llm-semantic-router/mmbert-32k-yarn
Key                                            | Status     | 
-----------------------------------------------+------------+-
model.layers.{0...21}.attn.rotary_emb.inv_freq | UNEXPECTED | 
decoder.bias                                   | UNEXPECTED | 
classifier.weight                              | MISSING    | 
classifier.bias                                | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 6,769,166 || all params: 314,310,172 || trainable%: 2.1537


In [10]:
MAX_LENGTH = 512

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

cols_to_keep = {"label", "input_ids", "attention_mask"}
train_tok = train_ds.map(tokenize_fn, batched=True,
    remove_columns=[c for c in train_ds.column_names if c not in cols_to_keep])
eval_tok = eval_ds.map(tokenize_fn, batched=True,
    remove_columns=[c for c in eval_ds.column_names if c not in cols_to_keep])

train_tok.set_format("torch")
eval_tok.set_format("torch")

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

In [11]:
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

training_args = TrainingArguments(
    output_dir="./mmbert32k-category-classifier-lora",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True,
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    dataloader_num_workers=2,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    compute_metrics=compute_metrics,
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss,Accuracy,F1 Weighted
200,1.493772,0.900813,0.728571,0.727105
400,0.882406,0.558783,0.800000,0.790095
600,0.654625,0.467119,0.885714,0.881818
800,0.327300,0.782493,0.800000,0.796155


Step,Training Loss,Validation Loss,Accuracy,F1 Weighted
200,1.493772,0.900813,0.728571,0.727105
400,0.882406,0.558783,0.800000,0.790095
600,0.654625,0.467119,0.885714,0.881818
800,0.327300,0.782493,0.800000,0.796155
1000,0.277892,0.602733,0.914286,0.913961
1200,0.131402,0.652460,0.900000,0.898593
1400,0.108092,0.642847,0.914286,0.913709
1600,0.006022,0.642007,0.914286,0.913709
1800,0.017852,0.646005,0.914286,0.913709


TrainOutput(global_step=1880, training_loss=0.6651068631955918, metrics={'train_runtime': 4137.7326, 'train_samples_per_second': 14.539, 'train_steps_per_second': 0.454, 'total_flos': 2.175271731462144e+16, 'train_loss': 0.6651068631955918, 'epoch': 5.0})

In [12]:
from sklearn.metrics import classification_report

predictions = trainer.predict(eval_tok)
preds = np.argmax(predictions.predictions, axis=-1)

print(classification_report(
    predictions.label_ids, preds,
    target_names=[IDX_TO_CATEGORY[i] for i in range(14)],
    digits=3
))

                  precision    recall  f1-score   support

         biology      1.000     1.000     1.000         5
        business      1.000     1.000     1.000         5
       chemistry      0.833     1.000     0.909         5
computer science      1.000     1.000     1.000         5
       economics      1.000     0.800     0.889         5
     engineering      1.000     0.600     0.750         5
          health      1.000     1.000     1.000         5
         history      1.000     1.000     1.000         5
             law      0.833     1.000     0.909         5
            math      1.000     0.800     0.889         5
           other      0.667     0.800     0.727         5
      philosophy      1.000     1.000     1.000         5
         physics      0.714     1.000     0.833         5
      psychology      1.000     0.800     0.889         5

        accuracy                          0.914        70
       macro avg      0.932     0.914     0.914        70
    weighted

In [13]:
import os, json, shutil

OUTPUT_DIR = "./mmbert32k-category-classifier-merged"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. 合并 LoRA 权重到基座模型（candle-binding 不支持运行时 LoRA）
merged_model = model.merge_and_unload()

# 2. 转为 float32（candle-binding 兼容性要求）
merged_model = merged_model.float()

# 3. 保存为 safetensors 格式
merged_model.save_pretrained(OUTPUT_DIR, safe_serialization=True, max_shard_size="5GB")

# 4. 保存 tokenizer
tokenizer.save_pretrained(OUTPUT_DIR)

# 5. 保存 category_mapping.json
category_mapping = {
    "category_to_idx": CATEGORY_TO_IDX,
    "idx_to_category": {str(v): k for k, v in CATEGORY_TO_IDX.items()}
}
with open(os.path.join(OUTPUT_DIR, "category_mapping.json"), "w") as f:
    json.dump(category_mapping, f, indent=2)

# 6. 保存 lora_config.json（训练元信息）
lora_meta = {"rank": 32, "alpha": 64, "dropout": 0.1,
             "target_modules": ["attn.Wqkv", "attn.Wo", "mlp.Wi", "mlp.Wo"]}
with open(os.path.join(OUTPUT_DIR, "lora_config.json"), "w") as f:
    json.dump(lora_meta, f)

# 7. 验证 config.json 与 candle-binding 的兼容性
with open(os.path.join(OUTPUT_DIR, "config.json"), "r") as f:
    config = json.load(f)

assert config["architectures"] == ["ModernBertForSequenceClassification"]
assert config["model_type"] == "modernbert"
assert "id2label" in config and len(config["id2label"]) == 14

config["dtype"] = "float32"
with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print(f"导出文件: {os.listdir(OUTPUT_DIR)}")

# 8. 打包下载
shutil.make_archive("mmbert32k-category-classifier-merged", "zip", OUTPUT_DIR)
print("请下载: mmbert32k-category-classifier-merged.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

导出文件: ['category_mapping.json', 'tokenizer_config.json', 'lora_config.json', 'tokenizer.json', 'config.json', 'model.safetensors']
请下载: mmbert32k-category-classifier-merged.zip
